# 01 Download: NHANES 1999-2018 + NCHS Linked Mortality Files

Pulls the NHANES exam/lab/questionnaire components needed for the cardiometabolic mortality
project, and the NCHS public-use Linked Mortality Files (NDI), for all 10 cycles 1999-2000
through 2017-2018.

**Scope**
- Components: `DEMO`, `BMX`, `BPX`, `GHB`, `DIQ`, `BPQ`, `MCQ`, `SMQ`, `GLU`, `TCHOL`, `HDL`, `TRIGLY`, `ALB_CR`, `BIOPRO`
- Mortality: `NHANES_<cycle>_MORT_2019_PUBLIC.dat`, follow-up through Dec 31 2019 (public-use)
- Output: raw `.xpt`/`.dat` under `data/raw/`, tidy `.parquet` under `data/parquet/`

**Known quirk handled here:** before the 2005-2006 cycle, several lab components used
different file stems than the harmonized names used from 2005 onward (e.g. fasting glucose
was `LAB10AM`/`L10AM_B`/`L10AM_C` before becoming `GLU_D` onward). The resolver below tries the
right candidate per cycle and logs what actually resolved, rather than assuming a name is right.

**Where files are saved:** in Colab, the first code cell mounts Google Drive (you'll be asked to
allow access) and writes everything into `My Drive/NHANES Cardiometabolic Mortality Project/data/`.
Run locally instead, it uses the project folder one level above `notebooks/`.

Check the manifest at the end: any `component` marked `missing` for `DEMO`, `BMX`, or `GHB` blocks
the cohort definition in notebook 02 and needs a look before moving on.

In [1]:
import requests, pandas as pd, pathlib, time
from io import BytesIO

try:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = pathlib.Path("/content/drive/MyDrive/NHANES Cardiometabolic Mortality Project")
except ImportError:
    ROOT = pathlib.Path.cwd().parent

assert (ROOT / "notebooks").exists(), f"Project folder not found at {ROOT}"
RAW_DIR = ROOT / "data" / "raw"
NHANES_DIR = RAW_DIR / "nhanes"
MORT_DIR = RAW_DIR / "mortality"
PARQUET_DIR = ROOT / "data" / "parquet"
for d in (NHANES_DIR, MORT_DIR, PARQUET_DIR):
    d.mkdir(parents=True, exist_ok=True)

HEADERS = {"User-Agent": "Mozilla/5.0 (research download; contact: ashiqurrahmankhan04@gmail.com)"}

Mounted at /content/drive


## Cycle map

Start year and NHANES file-name suffix per cycle.

In [2]:
CYCLES = {
    "1999-2000": {"year": 1999, "suffix": ""},
    "2001-2002": {"year": 2001, "suffix": "_B"},
    "2003-2004": {"year": 2003, "suffix": "_C"},
    "2005-2006": {"year": 2005, "suffix": "_D"},
    "2007-2008": {"year": 2007, "suffix": "_E"},
    "2009-2010": {"year": 2009, "suffix": "_F"},
    "2011-2012": {"year": 2011, "suffix": "_G"},
    "2013-2014": {"year": 2013, "suffix": "_H"},
    "2015-2016": {"year": 2015, "suffix": "_I"},
    "2017-2018": {"year": 2017, "suffix": "_J"},
}

## Component map

`legacy` lists the candidate file stems for cycles before 2005-2006; from 2005-2006 onward the
component's own name is used. Verified against the CDC variable lists:
- HbA1c: `LAB10`, `L10_B`, `L10_C`, then `GHB_D` onward
- HDL: inside the total cholesterol file (`LAB13`, `L13_B`, `L13_C`) until 2005, then its own `HDL_D` onward.
  For 1999-2004 the `HDL` parquet is therefore a second copy of the cholesterol file; notebook 02 only takes the HDL column from it.
- Serum creatinine (for eGFR): `LAB18`, `L40_B`, `L40_C`, then `BIOPRO_D` onward

In [3]:
COMPONENTS = {
    "DEMO":   {"label": "demographics",                 "legacy": ["DEMO"]},
    "BMX":    {"label": "body_measures",                 "legacy": ["BMX"]},
    "BPX":    {"label": "blood_pressure_exam",           "legacy": ["BPX"]},
    "GHB":    {"label": "glycohemoglobin",               "legacy": ["GHB", "L10", "LAB10"]},
    "DIQ":    {"label": "diabetes_questionnaire",        "legacy": ["DIQ"]},
    "BPQ":    {"label": "bp_cholesterol_questionnaire",  "legacy": ["BPQ"]},
    "MCQ":    {"label": "medical_conditions",            "legacy": ["MCQ"]},
    "SMQ":    {"label": "smoking",                       "legacy": ["SMQ"]},
    "GLU":    {"label": "fasting_glucose",               "legacy": ["GLU", "L10AM", "LAB10AM"]},
    "TCHOL":  {"label": "total_cholesterol",             "legacy": ["TCHOL", "L13", "LAB13"]},
    "HDL":    {"label": "hdl_cholesterol",               "legacy": ["L13", "LAB13"]},
    "TRIGLY": {"label": "triglycerides_ldl",              "legacy": ["TRIGLY", "L13AM", "LAB13AM"]},
    "ALB_CR": {"label": "albumin_creatinine",            "legacy": ["ALB_CR", "L16", "LAB16"]},
    "BIOPRO": {"label": "biochemistry_profile",          "legacy": ["L40", "LAB18"]},
}

In [4]:
def resolve_and_fetch(stem, cycle_key):
    year, suffix = CYCLES[cycle_key]["year"], CYCLES[cycle_key]["suffix"]
    candidates = COMPONENTS[stem]["legacy"] if year < 2005 else [stem]
    for name in candidates:
        url = f"https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/{year}/DataFiles/{name}{suffix}.xpt"
        r = requests.get(url, headers=HEADERS, timeout=30)
        if r.ok and r.content[:2] != b"<!":
            return url, r.content
    return None, None

## Download NHANES components, cycle by cycle

In [5]:
manifest = []
raw_frames = {stem: [] for stem in COMPONENTS}

for cycle_key in CYCLES:
    for stem in COMPONENTS:
        out_path = NHANES_DIR / cycle_key / f"{stem}.xpt"
        out_path.parent.mkdir(parents=True, exist_ok=True)
        url, content = resolve_and_fetch(stem, cycle_key)
        if content is None:
            manifest.append({"cycle": cycle_key, "component": stem, "status": "missing", "url": None, "n_rows": None})
            continue
        out_path.write_bytes(content)
        df = pd.read_sas(BytesIO(content), format="xport")
        df["cycle"] = cycle_key
        raw_frames[stem].append(df)
        manifest.append({"cycle": cycle_key, "component": stem, "status": "ok", "url": url, "n_rows": len(df)})
        time.sleep(0.5)

manifest_df = pd.DataFrame(manifest)
manifest_df.to_csv(RAW_DIR / "download_manifest.csv", index=False)
manifest_df[manifest_df["status"] == "missing"]

,cycle,component,status,url,n_rows


Review the row above before continuing. Any `DEMO`, `BMX`, or `GHB` cycle marked `missing`
blocks the cohort definition in notebook 02; a missing lab or questionnaire component for one
cycle is workable (that cycle just has more missingness on that variable, handled later by
multiple imputation in R).

In [6]:
for stem, frames in raw_frames.items():
    if frames:
        pd.concat(frames, ignore_index=True).to_parquet(PARQUET_DIR / f"{COMPONENTS[stem]['label']}.parquet", index=False)

## Download and parse the NCHS Linked Mortality Files

Public-use LMF, follow-up through Dec 31 2019. Fixed-width column positions are copied from
CDC's own sample R read-in program (`R_ReadInProgramAllSurveys.R`,
https://ftp.cdc.gov/pub/Health_Statistics/NCHS/datalinkage/linked_mortality/), NHANES version,
converted from 1-based inclusive to 0-based half-open for `pandas.read_fwf`.

`SEQN` is the join key back to every NHANES component above. `MORTSTAT` is the outcome
(0 = assumed alive, 1 = assumed deceased, missing = ineligible or under 18).
`PERMTH_INT`/`PERMTH_EXM` are follow-up time in months from interview/exam to death or Dec 31 2019.

In [7]:
MORT_COLSPECS = [(0, 6), (14, 15), (15, 16), (16, 19), (19, 20), (20, 21), (42, 45), (45, 48)]
MORT_NAMES = ["seqn", "eligstat", "mortstat", "ucod_leading", "diabetes", "hyperten", "permth_int", "permth_exm"]

mort_frames = []
for cycle_key in CYCLES:
    fname = f"NHANES_{cycle_key.replace('-', '_')}_MORT_2019_PUBLIC.dat"
    url = f"https://ftp.cdc.gov/pub/Health_Statistics/NCHS/datalinkage/linked_mortality/{fname}"
    r = requests.get(url, headers=HEADERS, timeout=30)
    if not r.ok:
        manifest.append({"cycle": cycle_key, "component": "MORTALITY", "status": "missing", "url": url, "n_rows": None})
        continue
    (MORT_DIR / fname).write_bytes(r.content)
    df = pd.read_fwf(BytesIO(r.content), colspecs=MORT_COLSPECS, names=MORT_NAMES, na_values=["."])
    df["cycle"] = cycle_key
    mort_frames.append(df)
    manifest.append({"cycle": cycle_key, "component": "MORTALITY", "status": "ok", "url": url, "n_rows": len(df)})

mortality = pd.concat(mort_frames, ignore_index=True)
mortality.to_parquet(PARQUET_DIR / "mortality.parquet", index=False)
pd.DataFrame(manifest).to_csv(RAW_DIR / "download_manifest.csv", index=False)
mortality.shape

(101316, 9)

## Next: notebook 02, data audit

Merge each cycle's components on `SEQN`, attach `mortality` on `SEQN` + `cycle`, check
`ELIGSTAT`/age-18 exclusions, missingness by cycle and variable, and settle the cohort
definition before any modeling.